In [ ]:
!pip install datasets gradio transformers[torch] accelerate -U evaluate

In [ ]:
from datasets import load_dataset

dataset = load_dataset('/content/drive/MyDrive/FYPData_5Labels')
dataset

In [ ]:
#dataset = dataset["train"].train_test_split(seed=42, shuffle=True, test_size=0.1)
#dataset

In [ ]:
dataset['train']['audio']

In [ ]:
import librosa
import numpy as np
import matplotlib.pyplot as plt
import librosa.display

In [ ]:
from scipy.signal import butter, lfilter

In [ ]:
id2label_fn = dataset["train"].features["label"].int2str
#Calling a label
#id2label_fn(dataset["train"][4]["label"])

In [ ]:
def plot_wave(audio, s, label):
  # Create time axis in seconds
  #y, s = librosa.load(audio['path'], sr=4000)

  time_axis = np.arange(0, len(y)) / s

  # Plot the waveform
  plt.figure(figsize=(10, 4))
  plt.plot(time_axis, y, color='b')
  plt.title(label)
  plt.xlabel('Time (s)')
  plt.ylabel('Amplitude')
  plt.grid(True)
  plt.show()

In [ ]:
len(dataset['train']['audio'][360]['array'])

In [ ]:
y, s = librosa.load(dataset['train']['audio'][360]['path'], sr=4000)
label = id2label_fn(dataset["train"][360]["label"])

plot_wave(y, s, label)

In [ ]:
from scipy import signal
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
#scipy.signal.butter(N, Wn, btype='low', analog=False, output='ba', fs=None)[source]

In [ ]:
#lowpass filter
sos = signal.butter(8, 64, 'low', fs=4000, output='sos')

y = dataset['train']['audio'][360]['array']
label = id2label_fn(dataset["train"][360]["label"])
filtered = signal.sosfilt(sos, y)

time_axis = np.arange(0, len(y)) / 4000

# Plot the waveform
plot_wave(filtered, 4000, label + ' filtered')

In [ ]:
#highpass filter
sos = signal.butter(8, 64, 'high', fs=4000, output='sos')

y = dataset['train']['audio'][360]['array']
label = id2label_fn(dataset["train"][360]["label"])
filtered = signal.sosfilt(sos, y)

time_axis = np.arange(0, len(y)) / 4000

# Plot the waveform
plot_wave(filtered, 4000, label + ' filtered')

In [ ]:
mean_value = np.mean(filtered)
print('Mean = ' + str(mean_value))
std_dev = np.std(filtered)
print('STD_DEV = ' + str(std_dev))

In [ ]:
normalized_audio = (filtered - mean_value) / std_dev

plot_wave(normalized_audio, 4000, label + ' normalized')

In [ ]:
normalized = []
labels = []

In [ ]:
len(dataset['train'])

In [ ]:
#filtering and normalizing entire dataset
for i in range(len(dataset['train'])):
  sos = signal.butter(8, 64, 'low', fs=4000, output='sos')

  y = dataset['train']['audio'][i]['array']
  label = id2label_fn(dataset["train"][i]["label"])
  labels.append(label)
  filtered = signal.sosfilt(sos, y)

  mean_value = np.mean(filtered)
  std_dev = np.std(filtered)

  normalized_audio = (filtered - mean_value) / std_dev
  normalized.append(normalized_audio)

In [ ]:
plot_wave(normalized[100], 400, labels[100])

In [ ]:
from IPython.display import Audio

t_sr = 4000
display(Audio(normalized[100][:t_sr * 5], rate=t_sr))

In [ ]:
import librosa
import librosa.display
import numpy as np
from scipy.signal import butter, lfilter
from IPython.display import Audio

In [ ]:
import soundfile as sf
import os

In [ ]:
# Iterate over the dataset and create folders based on label numbers
for i in range(len(normalized)):
    label = labels[i]
    audio_data = normalized[i]

    # Create a folder if it doesn't exist
    if(label == 'Atraining_artifact'):
      folder_path = '/content/drive/MyDrive/PreprocessedFYP_5Labels_Highpass/Artifact'

    elif(label == 'Atraining_extrahls'):
      folder_path = '/content/drive/MyDrive/PreprocessedFYP_5Labels_Highpass/Extrahystole'

    elif(label == 'Btraining_extrasystole'):
      folder_path = '/content/drive/MyDrive/PreprocessedFYP_5Labels_Highpass/Extrasystole'

    elif(label == 'Btraining_murmur'):
      folder_path = '/content/drive/MyDrive/PreprocessedFYP_5Labels_Highpass/murmur'

    elif(label == 'normal'):
      folder_path = '/content/drive/MyDrive/PreprocessedFYP_5Labels_Highpass/Normal'

    elif(label == 'Btraining_noisymurmur'):
      folder_path = '/content/drive/MyDrive/PreprocessedFYP_5Labels_Highpass/noisymurmur'

    os.makedirs(folder_path, exist_ok=True)
    save_path = os.path.join(folder_path, f'preprocessed_audio_{i}.wav')
    sf.write(save_path, audio_data, 4000)

In [ ]:
import gradio as gr

def generate_audio():
    example = normalized.shuffle()[0]
    audio = example["audio"]
    return (
        audio["sampling_rate"],
        audio["array"],
    ), id2label_fn(example["label"])


with gr.Blocks() as demo:
    with gr.Column():
        for _ in range(4):
            audio, label = generate_audio()
            output = gr.Audio(audio, label=label)

demo.launch(debug=True)

In [ ]:
!pip install transformers

In [ ]:
!pip install datasets gradio transformers[torch] accelerate -U evaluate

In [ ]:
from datasets import load_dataset

dataset = load_dataset('/content/drive/MyDrive/Combined Augmented + Original')
dataset

In [ ]:
dataset = dataset["train"].train_test_split(seed=42, shuffle=True, test_size=0.1)
dataset

In [ ]:
from transformers import AutoFeatureExtractor

model_id = "ntu-spml/distilhubert"
feature_extractor = AutoFeatureExtractor.from_pretrained(
    model_id, do_normalize=True, return_attention_mask=True
)

In [ ]:
sampling_rate = feature_extractor.sampling_rate
sampling_rate

In [ ]:
from datasets import Audio

dataset = dataset.cast_column("audio", Audio(sampling_rate=sampling_rate))

In [ ]:
import numpy as np

sample = dataset["train"][0]["audio"]

print(f"Mean: {np.mean(sample['array']):.3}, Variance: {np.var(sample['array']):.3}")

In [ ]:
inputs = feature_extractor(sample["array"], sampling_rate=sample["sampling_rate"])

print(f"inputs keys: {list(inputs.keys())}")

print(
    f"Mean: {np.mean(inputs['input_values']):.3}, Variance: {np.var(inputs['input_values']):.3}"
)

In [ ]:
max_duration = 5.0


def preprocess_function(examples):
    audio_arrays = [x["array"] for x in examples["audio"]]
    inputs = feature_extractor(
        audio_arrays,
        sampling_rate=feature_extractor.sampling_rate,
        max_length=int(feature_extractor.sampling_rate * max_duration),
        truncation=True,
        return_attention_mask=True,
    )
    return inputs

In [ ]:
dataset_encoded = dataset.map(
    preprocess_function,
    remove_columns=["audio"],
    batched=True,
    batch_size=100,
    num_proc=1,
)
dataset_encoded

In [ ]:
id2label_fn = dataset["train"].features["label"].int2str

In [ ]:
id2label = {
    str(i): id2label_fn(i)
    for i in range(len(dataset_encoded["train"].features["label"].names))
}
label2id = {v: k for k, v in id2label.items()}

id2label["1"]

In [ ]:
from transformers import AutoModelForAudioClassification

num_labels = len(id2label)

model = AutoModelForAudioClassification.from_pretrained(
    model_id,
    num_labels=num_labels,
    label2id=label2id,
    id2label=id2label,
)

In [ ]:
#[REMOVED_TOKEN]
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
!pip install transformers

In [ ]:
#73% accuracy, lr = 5e-5, epochs 25, max length = 10s
import transformers
from transformers import TrainingArguments

model_name = model_id.split("/")[-1]
batch_size = 8
gradient_accumulation_steps = 1
num_train_epochs = 10

training_args = TrainingArguments(
    f"{model_name}-finetuned-PASCAL_Dataset_Augmented",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=0.0001,
    per_device_train_batch_size=batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=num_train_epochs,
    warmup_ratio=0.1,
    logging_steps=5,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    fp16=True,
    push_to_hub=True,
    weight_decay = 0.01,
)

In [ ]:
import evaluate
import numpy as np

metric = evaluate.load("accuracy")


def compute_metrics(eval_pred):
    """Computes accuracy on a batch of predictions"""
    predictions = np.argmax(eval_pred.predictions, axis=1)
    return metric.compute(predictions=predictions, references=eval_pred.label_ids)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model,
    training_args,
    train_dataset=dataset_encoded["train"],
    eval_dataset=dataset_encoded["test"],
    tokenizer=feature_extractor,
    compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:
from transformers import pipeline

model_id = "arham061/distilhubert-finetuned-PASCAL_Dataset_Augmented"
pipe = pipeline("audio-classification", model=model_id)

In [ ]:
def classify_audio(filepath):
    preds = pipe(filepath)
    outputs = {}
    for p in preds:
        outputs[p["label"]] = p["score"]
    return outputs

In [ ]:
!pip install gradio

In [ ]:
import gradio as gr

demo = gr.Interface(
    fn=classify_audio, inputs=gr.Audio(type="filepath"), outputs="label"
)
demo.launch(debug=True)

In [ ]:
trainer.save_model(f"FYP_Model")